# TICKET MJD-505 

De: Paula Herrero · Marketing y tienda online
Me dice Elena que hay pedidos de la web que nunca llegaron al ERP. ¿Cuántos son, cuánto dinero su‐
man, de cuántos clientes y desde cuándo pasa?

In [2]:
import json
from pathlib import Path
from collections import Counter

RAIZ = Path.home() / "master-data-ia"
ruta = RAIZ / "datos" / "web" / "pedidos_web.json"
print("Ruta:", ruta, "| existe:", ruta.exists())

with open(ruta, encoding="utf-8") as f:
    web = json.load(f)

perdidos = [p for p in web["orders"] if p["status"] == "sync_error"]

def importe(pedido):
    lineas = sum(i["qty"] * i["unit_price"] * (1 - i["discount"] / 100) for i in pedido["items"])
    return lineas + pedido["shipping"]

total = sum(importe(p) for p in perdidos)
clientes = {p["customer"]["nif"] for p in perdidos}
por_mes = Counter(p["created_at"][:7] for p in perdidos)

print(f"Pedidos que no llegaron al ERP: {len(perdidos)}")
print(f"Importe: {total:,.2f} € (con portes)")
print(f"Clientes distintos: {len(clientes)}")
print(f"Primero: {min(p['created_at'] for p in perdidos)}")
print(f"Último: {max(p['created_at'] for p in perdidos)}")
print("Por mes:", dict(sorted(por_mes.items())))

Ruta: /Users/alvaroocrespoo/master-data-ia/datos/web/pedidos_web.json | existe: True
Pedidos que no llegaron al ERP: 31
Importe: 11,517.21 € (con portes)
Clientes distintos: 31
Primero: 2025-05-13T21:40:00Z
Último: 2026-08-31T23:24:00Z
Por mes: {'2025-05': 1, '2025-06': 1, '2025-07': 2, '2025-08': 1, '2025-09': 2, '2025-10': 4, '2025-11': 2, '2025-12': 2, '2026-01': 2, '2026-02': 3, '2026-03': 2, '2026-04': 2, '2026-05': 3, '2026-06': 2, '2026-08': 2}
